# Classification: AdaBoost Classifier

## Justification of Preprocessing Strategy

### Scale Invariance

AdaBoost (Adaptive Boosting) typically uses shallow Decision Trees, often called Decision Stumps, as its base estimators. Because these trees partition data based on feature thresholds rather than distance metrics, AdaBoost is invariant to the scale of the input features. Standardization or normalization will not affect the model's decision boundaries. Therefore, we will use the **Original Data** to maintain computational efficiency.

### Adaptive Learning Mechanism

The core strength of AdaBoost lies in its sequential learning process. It assigns weights to each training sample; in each iteration, it increases the weights of the samples that were incorrectly classified by the previous model. This forces the next weak learner to focus on the most difficult cases, such as patients with clinical values that are hard to distinguish between diabetic and non-diabetic.

## Experiment Design

We defined a tournament of 3 optimization levels to find the best balance between model complexity and learning speed:

- **Baseline**: default parameters (`n_estimators=50`, `learning_rate=1.0`) as per Scikit-Learn documentation.
- **GridSearchCV**: a systematic search over the number of estimators and the learning rate to identify the optimal ensemble size.
- **Optuna**: Bayesian optimization to fine-tune `learning_rate` and `n_estimators` simultaneously to maximize recall.

In [3]:
import pandas as pd
import numpy as np
import time
import mlflow
import optuna
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import recall_score, accuracy_score, f1_score

# MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Classification_AdaBoost")

<Experiment: artifact_location=('file:c:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente '
 'de '
 'Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/notebooks/Classification/EnsembleMethods/Boosting/AdaBoostClassifier/mlruns/9'), creation_time=1778150842712, experiment_id='9', last_update_time=1778150842712, lifecycle_stage='active', name='Classification_AdaBoost', tags={}, workspace='default'>

In [4]:
df = pd.read_csv("C:\\Users\\Tiago Silva\\Uni\\OneDrive - Universidade Portucalense\\Ambiente de Trabalho\\Uni\\3ano2sem\\LAD\\Grupo5_ProjetoLAD_Parte2\\TrabalhoLAD\\data\\diabetes_dataset_new_variables.csv")

categorical_cols = [
    'gender', 'ethnicity', 'smoking_status', 'education_level',
    'employment_status', 'age_groups', 'weight_status', 'income_level'
]

# One-Hot Encoding
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df_final.drop(["diagnosed_diabetes", "diabetes_stage"], axis=1)
y = df_final['diagnosed_diabetes']

# Split data (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def log_metrics(y_true, y_pred, duration):
    """Log performance metrics to MLflow"""
    mlflow.log_metric("recall", recall_score(y_true, y_pred))
    mlflow.log_metric("accuracy", accuracy_score(y_true, y_pred))
    mlflow.log_metric("f1", f1_score(y_true, y_pred))
    mlflow.log_metric("fit_time", duration)

# ---------------------------------------------------------
# RUN 1: BASELINE 
# ---------------------------------------------------------
with mlflow.start_run(run_name="AdaBoost_Baseline"):
    ada_base = AdaBoostClassifier(random_state=42)
    
    start_time = time.time()
    ada_base.fit(X_train, y_train)
    duration = time.time() - start_time
    
    y_pred = ada_base.predict(X_test)
    
    mlflow.log_params(ada_base.get_params())
    mlflow.log_param("optimization", "none")
    log_metrics(y_test, y_pred, duration)

# ---------------------------------------------------------
# RUN 2: GRIDSEARCHCV 
# ---------------------------------------------------------
with mlflow.start_run(run_name="AdaBoost_GridSearch"):
    # Testing different combinations of estimators and learning speed
    param_grid = {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 1.0]
    }
    
    grid = GridSearchCV(
        AdaBoostClassifier(random_state=42),
        param_grid, cv=3, scoring='recall', n_jobs=-1
    )
    
    start_time = time.time()
    grid.fit(X_train, y_train)
    duration = time.time() - start_time
    
    y_pred_grid = grid.best_estimator_.predict(X_test)
    
    mlflow.log_params(grid.best_params_)
    mlflow.log_param("optimization", "GridSearchCV")
    log_metrics(y_test, y_pred_grid, duration)

# ---------------------------------------------------------
# RUN 3: OPTUNA 
# ---------------------------------------------------------
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 10, 300),
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 1.0, log=True)
    }
    
    model = AdaBoostClassifier(**params)
    # Using 3-fold CV for efficiency
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='recall', n_jobs=-1).mean()
    return score

with mlflow.start_run(run_name="AdaBoost_Optuna"):
    study = optuna.create_study(direction="maximize")
    start_time = time.time()
    study.optimize(objective, n_trials=15) 
    duration = time.time() - start_time
    
    # Train final champion model
    best_ada = AdaBoostClassifier(**study.best_params, random_state=42)
    best_ada.fit(X_train, y_train)
    
    mlflow.log_params(study.best_params)
    mlflow.log_param("optimization", "optuna")
    log_metrics(y_test, best_ada.predict(X_test), duration)

[I 2026-05-07 11:50:18,383] A new study created in memory with name: no-name-6a215bae-7ced-49b3-963c-88abbf5c6702
[I 2026-05-07 11:50:31,208] Trial 0 finished with value: 0.8520979813842532 and parameters: {'n_estimators': 111, 'learning_rate': 0.26385751925968887}. Best is trial 0 with value: 0.8520979813842532.
[I 2026-05-07 11:50:55,630] Trial 1 finished with value: 0.8520979813842532 and parameters: {'n_estimators': 226, 'learning_rate': 0.009170585656983376}. Best is trial 0 with value: 0.8520979813842532.
[I 2026-05-07 11:51:09,353] Trial 2 finished with value: 0.8520979813842532 and parameters: {'n_estimators': 119, 'learning_rate': 0.2245888752648399}. Best is trial 0 with value: 0.8520979813842532.
[I 2026-05-07 11:51:14,152] Trial 3 finished with value: 0.8520979813842532 and parameters: {'n_estimators': 40, 'learning_rate': 0.0061526285222243335}. Best is trial 0 with value: 0.8520979813842532.
[I 2026-05-07 11:51:35,373] Trial 4 finished with value: 0.8520979813842532 and p

## Runs Summary

| Run | Optimization | n_estimators | learning_rate | Accuracy | F1 | Recall | Fit Time |
|---|---|---:|---:|---:|---:|---:|---:|
| AdaBoost_Baseline | none | 50 | 1.0 | 0.9199 | 0.9284757568 | 0.8665 | 6.995s |
| AdaBoost_GridSearch | GridSearchCV | 50 | 1.0 | 0.9199 | 0.9284757568 | 0.8665 | 78.807s |
| AdaBoost_Optuna | optuna | 297 | 0.9774035054 | 0.9199 | 0.9284757568 | 0.8665 | 338.131s |

### Additional logged parameters
- `random_state = 42` where set in code
- `base_estimator`: Decision stump / default (quando aplicável)
- Parâmetros otimizados: `n_estimators`, `learning_rate`

## Best Run Justification for Streamlit

All runs have identical metrics (Accuracy = 0.9199, F1 = 0.92848, Recall = 0.8665). Therefore, the decision is based on secondary criteria: simplicity, training time, and inference cost.

- **AdaBoost_Baseline** uses `n_estimators=50` and `learning_rate=1.0` and has the lowest training time (~7s).
- **AdaBoost_GridSearch** also found `n_estimators=50` and `learning_rate=1.0`, but its training cost was much higher (≈78.8s) with no improvement in metrics.
- **AdaBoost_Optuna** found a larger ensemble (`n_estimators=297`, `learning_rate≈0.9774`) with identical metrics, but with a much higher training cost (≈338s) and an expected higher inference cost.

Recommendations for Streamlit:
- Choose **AdaBoost_Baseline**: same predictive performance, lower training cost, and lower inference cost (fewer estimators → faster predictions). It is the simplest and most efficient option for production.
- If the goal is to further reduce inference time per record, consider models with far fewer estimators or lighter base learners (for example, an optimized `DecisionTreeClassifier`).